In [5]:
import json
import pyLDAvis
import pyLDAvis.lda_model
pyLDAvis.enable_notebook()

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [6]:
def load_narratives(narrative_id, gender_included):
    with open(f'../data/narratives/synthetic_narratives_{narrative_id}_gender_{gender_included}.json') as f:
      narratives = json.load(f)
    return narratives

def load_profiles():
    with open('../data/synthetic_profiles.json') as f:
      synthetic_profiles = json.load(f)
    return synthetic_profiles

In [7]:
synthetic_profiles = load_profiles()
profiles_M = [item['id'] for item in synthetic_profiles if item['gender'] == "male"]
profiles_F = [item['id'] for item in synthetic_profiles if item['gender'] == "female"]

In [8]:
data = load_narratives(1, True)
narratives_raw = [item['narrative_text'] for item in data if 'narrative_text' in item]

narratives_raw_M = [item['narrative_text'] for item in data if 'narrative_text' in item and item['profile_id'] in profiles_M]
narratives_raw_F = [item['narrative_text'] for item in data if 'narrative_text' in item and item['profile_id'] in profiles_F]

In [9]:
def LDA_model(docs_raw, n_topics, stop_words = ENGLISH_STOP_WORDS):
    tf_vectorizer = CountVectorizer(strip_accents = 'unicode',
                                stop_words = stop_words,
                                lowercase = True,
                                token_pattern = r'\b[a-zA-Z]{3,}\b',
                                max_df = 0.5,  # exclude words with a relative document frequency greater than 50%
                                min_df = 10    # exclude tokens that occur less than 10 times
                                )
    docs_vectorized = tf_vectorizer.fit_transform(docs_raw)
    lda_tf = LatentDirichletAllocation(n_components=n_topics, random_state=0,verbose=1, max_iter=10)
    lda_tf.fit(docs_vectorized)
    return lda_tf, docs_vectorized, tf_vectorizer

In [10]:
def print_most_salient_topics(panel, n_top_topics=5):
    topic_info = panel.topic_info
    # Filter out the individual terms to get the topics
    topics = topic_info[topic_info.Category == 'Topic']
    # Sort topics by saliency
    salient_topics = topics.sort_values(by='Total', ascending=False).head(n_top_topics)
    for idx, row in salient_topics.iterrows():
        topic_id = int(row['Category'].split()[-1])
        print(f"Topic {topic_id}:")
        # Extract the top words for this topic
        term_list = panel.token_table[panel.token_table.Topic == topic_id]
        term_list = term_list.sort_values(by='Freq', ascending=False)
        terms = term_list.Term.tolist()
        print(f"  Top terms: {terms[:5]}") 

In [11]:
def evaluate_lda_model(data_vectorized, lda_model):
  # Log Likelihood: Higher the better
  print("Log Likelihood: ", lda_model.score(data_vectorized))

  # Perplexity: Lower the better. Perplexity = exp(-1. * log-likelihood per word)
  print("Perplexity: ", lda_model.perplexity(data_vectorized))

  # See model parameters
  print(lda_model.get_params())

In [12]:
custom_stop_words = list(['agnes', 'amelia', 'maya', 'sarah', 'eleanor', 'anya'])
stop_words_F = list(ENGLISH_STOP_WORDS.union(custom_stop_words))

lda_tf_F, docs_vectorized_F, tf_vectorizer_F = LDA_model(narratives_raw_F, 6, stop_words_F)
evaluate_lda_model(docs_vectorized_F, lda_tf_F)

iteration: 1 of max_iter: 10
iteration: 2 of max_iter: 10
iteration: 3 of max_iter: 10
iteration: 4 of max_iter: 10
iteration: 5 of max_iter: 10
iteration: 6 of max_iter: 10
iteration: 7 of max_iter: 10
iteration: 8 of max_iter: 10
iteration: 9 of max_iter: 10
iteration: 10 of max_iter: 10
Log Likelihood:  -2289235.424504802
Perplexity:  1665.6778155651016
{'batch_size': 128, 'doc_topic_prior': None, 'evaluate_every': -1, 'learning_decay': 0.7, 'learning_method': 'batch', 'learning_offset': 10.0, 'max_doc_update_iter': 100, 'max_iter': 10, 'mean_change_tol': 0.001, 'n_components': 6, 'n_jobs': None, 'perp_tol': 0.1, 'random_state': 0, 'topic_word_prior': None, 'total_samples': 1000000.0, 'verbose': 1}


In [13]:
panel_F = pyLDAvis.lda_model.prepare(lda_tf_F, docs_vectorized_F, tf_vectorizer_F)
panel_F

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5     -0.003559  0.028509       1        1  35.454419
1     -0.132958 -0.046667       2        1  20.424880
2     -0.112738  0.074605       3        1  15.733358
4      0.118668  0.093083       4        1  11.172121
3      0.050612 -0.109666       5        1   8.815499
0      0.079974 -0.039864       6        1   8.399723, topic_info=              Term         Freq        Total Category  logprob  loglift
1261       factory   567.000000   567.000000  Default  30.0000  30.0000
1283          farm   624.000000   624.000000  Default  29.0000  29.0000
1608         hands  1307.000000  1307.000000  Default  28.0000  28.0000
3797      workshop   472.000000   472.000000  Default  27.0000  27.0000
1926          land   518.000000   518.000000  Default  26.0000  26.0000
...            ...          ...          ...      ...      ...      ...
2937        school   106.031785   930.166031   Topic6  -5.4991   0.3053
2165          mind   100.931074   674.168355   Topic6  -5.5484   0.5779
2650       purpose   103.107282  1008.264715   Topic6  -5.5271   0.1968
2909  satisfaction   100.056266   758.099789   Topic6  -5.5571   0.4519
3640       vibrant    99.203071  1092.558722   Topic6  -5.5657   0.0779

[478 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
10        1  0.099618  academia
10        3  0.827594  academia
10        5  0.061303  academia
10        6  0.015326  academia
11        1  0.104166  academic
...     ...       ...       ...
3824      2  0.047603     young
3824      3  0.147477     young
3824      4  0.097073     young
3824      5  0.098940     young
3824      6  0.064404     young

[1265 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[6, 2, 3, 5, 4, 1])

In [14]:
custom_stop_words = list(['arthur', 'silas', 'elias', 'ethan', 'liam', 'mark', 'michael'])
stop_words_M = list(ENGLISH_STOP_WORDS.union(custom_stop_words))

lda_tf_M, docs_vectorized_M, tf_vectorizer_M = LDA_model(narratives_raw_M, 6, stop_words_M)
evaluate_lda_model(docs_vectorized_M, lda_tf_M)

iteration: 1 of max_iter: 10
iteration: 2 of max_iter: 10
iteration: 3 of max_iter: 10
iteration: 4 of max_iter: 10
iteration: 5 of max_iter: 10
iteration: 6 of max_iter: 10
iteration: 7 of max_iter: 10
iteration: 8 of max_iter: 10
iteration: 9 of max_iter: 10
iteration: 10 of max_iter: 10
Log Likelihood:  -2297857.303291156
Perplexity:  1752.2404972622855
{'batch_size': 128, 'doc_topic_prior': None, 'evaluate_every': -1, 'learning_decay': 0.7, 'learning_method': 'batch', 'learning_offset': 10.0, 'max_doc_update_iter': 100, 'max_iter': 10, 'mean_change_tol': 0.001, 'n_components': 6, 'n_jobs': None, 'perp_tol': 0.1, 'random_state': 0, 'topic_word_prior': None, 'total_samples': 1000000.0, 'verbose': 1}


In [15]:
pyLDAvis.lda_model.prepare(lda_tf_M, docs_vectorized_M, tf_vectorizer_M)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1      0.079200 -0.019557       1        1  29.159092
0     -0.078367 -0.036875       2        1  24.118542
2     -0.004679  0.026105       3        1  14.046753
3      0.134240 -0.078491       4        1  13.209295
5     -0.142205 -0.052154       5        1  11.485056
4      0.011811  0.160972       6        1   7.981260, topic_info=         Term        Freq        Total Category  logprob  loglift
1965     land  645.000000   645.000000  Default  30.0000  30.0000
1293     farm  621.000000   621.000000  Default  29.0000  29.0000
1270  factory  723.000000   723.000000  Default  28.0000  28.0000
1080    earth  474.000000   474.000000  Default  27.0000  27.0000
3876     wood  632.000000   632.000000  Default  26.0000  26.0000
...       ...         ...          ...      ...      ...      ...
3927    young   94.885139   812.804564   Topic6  -5.5560   0.3803
472    career   93.832719   814.231540   Topic6  -5.5672   0.3673
2074     long   97.316542  1070.213169   Topic6  -5.5307   0.1304
1320     felt   91.084076  1038.726012   Topic6  -5.5969   0.0941
1719  history   83.440165   581.999326   Topic6  -5.6846   0.5857

[466 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
8         1  0.036978  academia
8         2  0.044373  academia
8         3  0.036978  academia
8         4  0.828296  academia
8         5  0.029582  academia
...     ...       ...       ...
3927      2  0.259595     young
3927      3  0.102116     young
3927      4  0.150098     young
3927      5  0.141485     young
3927      6  0.116879     young

[1276 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 1, 3, 4, 6, 5])